## 1. Inicialización y Análisis Exploratorio Inicial

En esta primera etapa, configuramos el entorno de trabajo importando las librerías fundamentales para la manipulación de datos (`pandas`, `numpy`), la construcción del pipeline de Machine Learning (`scikit-learn`) y nuestro modelo predictivo (`xgboost`).

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
import joblib

# 1. Carga de datos 
df = pd.read_csv('../data/listings.csv')

# 2. Estructura
print("Dimensiones del dataset:", df.shape)
print("\nTipos de datos y valores nulos:")
df.info()


Dimensiones del dataset: (31430, 90)

Tipos de datos y valores nulos:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31430 entries, 0 to 31429
Data columns (total 90 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            31430 non-null  int64  
 1   listing_url                                   31430 non-null  object 
 2   scrape_id                                     31430 non-null  int64  
 3   last_scraped                                  31430 non-null  object 
 4   source                                        31430 non-null  object 
 5   name                                          31430 non-null  object 
 6   description                                   30729 non-null  object 
 7   neighborhood_overview                         0 non-null      float64
 8   picture_url                                   31430 non-null  obje

## 2. Limpieza de Datos (ETL)

Con base en el análisis exploratorio, ejecutamos una limpieza exhaustiva centrada en retener exclusivamente las características físicas y geográficas. Depuramos valores nulos en la variable objetivo (`price`), descartamos metadatos irrelevantes e imputamos los datos faltantes en métricas de capacidad (como `bedrooms` y `bathrooms`). Finalmente, aplicamos un filtrado de outliers y extraemos variables binarias desde las amenidades, preparando un dataset puramente hedónico.

In [2]:
# --- 1. SELECCIÓN PURAMENTE FÍSICA Y GEOGRÁFICA ---

columnas_utiles = [
    'price', 'neighbourhood_cleansed', 'room_type', 'accommodates',
    'bathrooms', 'bedrooms', 'beds', 'amenities', 'latitude', 'longitude',
    'minimum_nights'
]

df_clean = df[columnas_utiles].copy()

# --- 2. LIMPIEZA DE LA VARIABLE OBJETIVO (PRECIO) ---
df_clean = df_clean.dropna(subset=['price'])
df_clean['price'] = df_clean['price'].replace('[\$,]', '', regex=True).astype(float)

# --- 3. IMPUTACIÓN DE FALTANTES FÍSICOS ---
# Si no especifica cuartos/camas/baños, asumimos el mínimo funcional (1)
df_clean['bedrooms'] = df_clean['bedrooms'].fillna(1)
df_clean['beds'] = df_clean['beds'].fillna(1)
df_clean['bathrooms'] = df_clean['bathrooms'].fillna(1)

# --- 4. EXTRACCIÓN DE AMENIDADES (FEATURE ENGINEERING) ---
df_clean['amenities'] = df_clean['amenities'].str.lower()
df_clean['has_pool'] = df_clean['amenities'].str.contains('pool').astype(int)
df_clean['has_ac'] = df_clean['amenities'].str.contains('air conditioning|ac').astype(int)
df_clean['has_parking'] = df_clean['amenities'].str.contains('parking').astype(int)
df_clean['has_wifi'] = df_clean['amenities'].str.contains('wifi').astype(int)
df_clean = df_clean.drop(columns=['amenities'])

# --- 5. FILTRO DE OUTLIERS ---
limite_superior = df_clean['price'].quantile(0.95)
limite_inferior = 150 
df_clean = df_clean[(df_clean['price'] >= limite_inferior) & (df_clean['price'] <= limite_superior)]

print(f"Dataset enfocado en características físicas: {df_clean.shape[0]} filas")

# --- 6. PREPARACIÓN DE X e y ---
X = df_clean.drop(columns=['price'])
y = df_clean['price']

Dataset enfocado en características físicas: 28151 filas


## 3. Construcción del Pipeline y Entrenamiento del Modelo

Para asegurar la reproducibilidad y evitar la fuga de datos (*data leakage*), encapsulamos el preprocesamiento y el algoritmo predictivo en un `Pipeline` de Scikit-Learn. Estandarizamos las variables numéricas con `StandardScaler`, binarizamos las categóricas mediante `OneHotEncoder` y entrenamos un modelo `XGBRegressor` envuelto en una transformación logarítmica (`TransformedTargetRegressor`) para manejar eficazmente la distribución asimétrica positiva de los precios.

In [3]:
import numpy as np
from sklearn.compose import TransformedTargetRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
import joblib

# Separación de datos (por si no lo ejecutaste en la celda anterior)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- 1. DEFINICIÓN DE VARIABLES (Enfoque Hedónico/Físico) ---
features_numericas = [
    'latitude', 'longitude', 'accommodates', 'bathrooms', 
    'bedrooms', 'beds', 'minimum_nights', 
    'has_pool', 'has_ac', 'has_parking', 'has_wifi'
]

features_categoricas = ['neighbourhood_cleansed', 'room_type']

# --- 2. TRANSFORMADORES ---
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, features_numericas),
        ('cat', categorical_transformer, features_categoricas)
    ])

# --- 3. MODELO OPTIMIZADO (Con transformación logarítmica) ---
xgb_base = XGBRegressor(
    n_estimators=300,        
    learning_rate=0.05,      
    max_depth=7,             
    subsample=0.8,           
    colsample_bytree=0.8,    
    random_state=42
)

log_target_model = TransformedTargetRegressor(
    regressor=xgb_base,
    func=np.log1p,         
    inverse_func=np.expm1  
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', log_target_model)
])

# --- 4. ENTRENAMIENTO Y GUARDADO ---
print("Entrenando modelo de precios basado en características físicas...")
pipeline.fit(X_train, y_train)
print("Entrenamiento completado.")

joblib.dump(pipeline, '../models/airbnb_pricing_pipeline.pkl')
print("Pipeline exportado a '../models/airbnb_pricing_pipeline.pkl'")

Entrenando modelo de precios basado en características físicas...
Entrenamiento completado.
Pipeline exportado a '../models/airbnb_pricing_pipeline.pkl'


## 4. Evaluación del Modelo (Métricas de Regresión)

Al tratarse de una predicción numérica continua, evaluamos el rendimiento del modelo utilizando el Error Absoluto Medio (MAE) y la Raíz del Error Cuadrático Medio (RMSE). El MAE nos indica la desviación promedio monetaria, calculada como $MAE = \frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i|$, lo cual resulta altamente interpretable para el negocio. Por su parte, el RMSE, definido como $RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$, penaliza severamente los errores de gran magnitud, asegurando la robustez de nuestra sugerencia de precios ante predicciones extremas.

In [4]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

print("Generando predicciones sobre el conjunto de prueba...")
y_pred = pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print("-" * 30)
print("RESULTADOS DE LA EVALUACIÓN")
print("-" * 30)
print(f"MAE  (Error Absoluto Medio): ${mae:.2f}")
print(f"RMSE (Error Cuadrático Medio): ${rmse:.2f}")
print(f"Precio promedio en test:     ${y_test.mean():.2f}")
print("-" * 30)

margen_error_pct = (mae / y_test.mean()) * 100
print(f"El modelo tiene un margen de error promedio del {margen_error_pct:.1f}% respecto al precio medio.")

Generando predicciones sobre el conjunto de prueba...
------------------------------
RESULTADOS DE LA EVALUACIÓN
------------------------------
MAE  (Error Absoluto Medio): $621.61
RMSE (Error Cuadrático Medio): $996.29
Precio promedio en test:     $2027.41
------------------------------
El modelo tiene un margen de error promedio del 30.7% respecto al precio medio.
